# Catch Joe — ML Workflow

Binary **target-user detection** from web session data.  
Three models compared end-to-end with MLflow tracking.

| | |
|---|---|
| **Primary metric** | PR-AUC (class-imbalance aware) |
| **Positive class** | sessions from `TARGET_USER_ID` |
| **Negative class** | all other users |
| **Split strategy** | Time-based (preferred) / Stratified (fallback) |

---
**Sections**  
0 · Setup & Configuration  
1 · Data Loading & EDA  
2 · Model 1: CatBoost (top-K domain sweep)  
3 · Model 2: TF-IDF + Logistic Regression (max_features sweep)  
4 · Model 3: Siamese Encoder (contrastive)  
5 · Comparison & Recommendation

## 0 · Setup & Configuration

In [1]:
import sys, os, json, pickle, warnings, tempfile, gc
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.catboost as mlflow_catboost
import mlflow.sklearn as mlflow_sklearn

# Make catch_joe importable when running from notebooks/
REPO_ROOT = Path('.').resolve().parent
SRC_PATH  = REPO_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from catch_joe.data import (
    load_sessions, validate_schema, create_target,
    time_based_split, stratified_split, get_user_stats,
)
from catch_joe.features import (
    extract_session_stats, get_top_k_domains, build_tree_features,
    build_site_indicators, METADATA_CAT_COLS, METADATA_NUM_COLS,
)
from catch_joe.evaluation import (
    compute_all_metrics, plot_confusion_matrix, plot_pr_curve,
    compare_models, log_run_to_mlflow,
)
from catch_joe.modeling import (
    train_catboost, train_tfidf_lr,
    build_site_vocab, train_siamese,
    encode_sessions, predict_siamese_scores,
    SIAMESE_NUM_COLS,
)

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100
print('Imports OK')


Imports OK


In [2]:
# ── Configuration — edit these values ────────────────────────────────────
TARGET_USER_ID    = 0          # <<< user you want to detect
DATA_PATH         = '../data/raw/dataset.json'
MLFLOW_EXPERIMENT = 'catch_joe_detection'
SPLIT_STRATEGY    = 'stratified'     # 'time' or 'stratified'
TEST_RATIO        = 0.20
RANDOM_STATE      = 42

# Explicit tracking URI anchored to repo root — CWD-independent
MLFLOW_DB = f"sqlite:///{REPO_ROOT}/mlflow.db"
mlflow.set_tracking_uri(MLFLOW_DB)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print(f'MLflow experiment : {MLFLOW_EXPERIMENT}')
print(f'MLflow tracking   : {MLFLOW_DB}')
print(f'Target user ID    : {TARGET_USER_ID}')
print(f'Split strategy    : {SPLIT_STRATEGY}')


MLflow experiment : catch_joe_detection
MLflow tracking   : sqlite:////home/christian/toptal/catch_joe/Christian-jesus-flores-gomez/catch_joe/mlflow.db
Target user ID    : 0
Split strategy    : stratified


## 1 · Data Loading & EDA

In [3]:
print('Loading sessions ...')
df_raw = load_sessions(DATA_PATH)
validate_schema(df_raw)

print(f'Total sessions : {len(df_raw):,}')
print(f'Unique users   : {df_raw["user_id"].nunique()}')
print(f'Date range     : {df_raw["date"].min()} → {df_raw["date"].max()}')
df_raw.head(2)

Loading sessions ...
Total sessions : 160,000
Unique users   : 200
Date range     : 2016-01-14 → 2019-06-18


,browser,os,locale,gender,country,city,sites_visited,site_lengths,time,date,user_id,datetime
0,Chrome,Windows 8,de-DE,m,Canada,Toronto,"[lenta.ru, lenta.ru, vk.com, lenta.ru, wikiped...","[296, 69, 94, 129, 70, 120, 54, 213, 140, 166,...",03:57:00,2016-08-14,164,2016-08-14 03:57:00
1,Chrome,Windows 10,pt-PT,f,Netherlands,Amsterdam,"[windowsupdate.com, amazon.com, live.com, akam...","[56, 413, 64, 60, 99, 63, 50, 41, 60, 55, 252,...",13:52:00,2016-05-31,99,2016-05-31 13:52:00


In [4]:
# Sessions per user — pick a well-represented TARGET_USER_ID
user_stats = get_user_stats(df_raw)
print('Top-15 users by session count:')
print(user_stats.head(15).to_string(index=False))

target_rows = user_stats[user_stats['user_id'] == TARGET_USER_ID]
if target_rows.empty:
    print(f'\nWARNING: user_id={TARGET_USER_ID} not found.')
    print('Set TARGET_USER_ID to one of the users listed above.')
else:
    n_sess = target_rows.iloc[0]['session_count']
    print(f'\nTarget user {TARGET_USER_ID} has {n_sess:,} sessions')

Top-15 users by session count:
 user_id  session_count
       0            800
       1            800
       2            800
       3            800
       4            800
       5            800
       6            800
       7            800
       8            800
       9            800
      10            800
      11            800
      12            800
      13            800
      14            800

Target user 0 has 800 sessions


In [5]:
# Create binary target label and engineer session features
df = create_target(df_raw, TARGET_USER_ID)
df = extract_session_stats(df)

pos = int(df['is_target_user'].sum())
neg = len(df) - pos
print(f'Positive (target)  : {pos:,}  ({100*pos/len(df):.2f}%)')
print(f'Negative (others)  : {neg:,}  ({100*neg/len(df):.2f}%)')
print(f'Imbalance ratio    : 1:{neg//max(pos,1)}')

Positive (target)  : 800  (0.50%)
Negative (others)  : 159,200  (99.50%)
Imbalance ratio    : 1:199


In [6]:
# Train / validation split
if SPLIT_STRATEGY == 'time':
    df_train, df_val = time_based_split(df, test_ratio=TEST_RATIO)
    split_note = 'Time-based split (temporal ordering preserved)'
else:
    df_train, df_val = stratified_split(df, test_ratio=TEST_RATIO,
                                         random_state=RANDOM_STATE)
    split_note = 'Stratified split  *** ignores temporal ordering ***'

y_train = df_train['is_target_user'].values
y_val   = df_val['is_target_user'].values

print(split_note)
print(f'Train : {len(df_train):,} sessions  (pos={int(y_train.sum())})')
print(f'Val   : {len(df_val):,} sessions  (pos={int(y_val.sum())})')

Stratified split  *** ignores temporal ordering ***
Train : 128,000 sessions  (pos=640)
Val   : 32,000 sessions  (pos=160)


In [7]:
# Quick EDA: metadata distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['browser', 'os', 'gender']):
    counts = df[col].value_counts().head(10)
    sns.barplot(x=counts.values, y=counts.index, ax=ax)
    ax.set_title(col)
plt.suptitle('Top metadata values (all sessions)', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

# Top-20 sites overall
from collections import Counter
site_counts = Counter()
for sites in df['sites_visited']:
    site_counts.update(sites)
top20 = pd.Series(dict(site_counts.most_common(20)))
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=top20.values, y=top20.index, ax=ax)
ax.set_title('Top-20 most visited domains')
plt.tight_layout()
plt.show()

## 2 · Model 1: CatBoost (tree-based tabular)

- All metadata columns handled as native categoricals
- Top-K binary domain indicators added as boolean features
- `auto_class_weights='Balanced'` handles class imbalance
- Sweep: K ∈ {500, 1000, 5000}

In [8]:
# CatBoost sweep — top_k ∈ {500, 1000, 5000}
# Each run is fully logged to MLflow; no in-memory results dicts.
for top_k in [500, 1000, 5000]:
    print(f'\n{"="*60}')
    print(f'CatBoost  top_k={top_k}')
    print('='*60)

    X_train_cb, X_val_cb, cat_idx, feat_names = build_tree_features(
        df_train, df_val, top_k=top_k
    )

    with mlflow.start_run(run_name=f'catboost_top{top_k}'):
        model_cb = train_catboost(
            X_train_cb, y_train, X_val_cb, y_val,
            cat_feature_indices=cat_idx,
            params={'iterations': 500, 'learning_rate': 0.05, 'depth': 6},
        )

        y_score_cb = model_cb.predict_proba(X_val_cb)[:, 1]
        y_pred_cb  = (y_score_cb >= 0.5).astype(int)

        k_eval  = min(500, max(1, int(y_val.sum())))
        metrics = compute_all_metrics(y_val, y_score_cb, k=k_eval)

        feat_imp = pd.DataFrame({
            'feature':    feat_names,
            'importance': model_cb.get_feature_importance(),
        }).sort_values('importance', ascending=False).reset_index(drop=True)

        fig_cm = plot_confusion_matrix(y_val, y_pred_cb, title=f'CatBoost K={top_k}')
        fig_pr = plot_pr_curve(y_val, y_score_cb, label=f'CatBoost K={top_k}')

        log_run_to_mlflow(
            params={
                'model_type':       'catboost',
                'target_user_id':   TARGET_USER_ID,
                'split_strategy':   SPLIT_STRATEGY,
                'top_k':            top_k,
                'iterations':       500,
                'learning_rate':    0.05,
                'depth':            6,
                'train_size':       len(df_train),
                'val_size':         len(df_val),
                'n_positive_train': int(y_train.sum()),
                'n_negative_train': int(len(y_train) - y_train.sum()),
            },
            metrics=metrics,
            figures={'confusion_matrix': fig_cm, 'pr_curve': fig_pr},
            model=model_cb,
            model_name='model',
            extra_artifacts={
                'feature_importance.csv': feat_imp,
                'top_domains':            get_top_k_domains(df_train, top_k),
                'feature_names':          feat_names,
            },
            model_flavor='catboost',
        )

    del X_train_cb, X_val_cb
    gc.collect()

    print(f'PR-AUC={metrics["pr_auc"]:.4f} | '
          f'ROC-AUC={metrics["roc_auc"]:.4f} | '
          f'P={metrics["precision"]:.3f} '
          f'R={metrics["recall"]:.3f} '
          f'F1={metrics["f1"]:.3f}')



CatBoost  top_k=500
0:	test: 0.9801023	best: 0.9801023 (0)	total: 312ms	remaining: 2m 35s
100:	test: 0.9974270	best: 0.9974270 (100)	total: 8.46s	remaining: 33.4s
200:	test: 0.9975063	best: 0.9975108 (175)	total: 14.2s	remaining: 21.2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9975151146
bestIteration = 214

Shrink model to first 215 iterations.


2026/06/09 18:56:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 18:56:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


PR-AUC=0.7045 | ROC-AUC=0.9975 | P=0.285 R=1.000 F1=0.443

CatBoost  top_k=1000
0:	test: 0.9446743	best: 0.9446743 (0)	total: 69.8ms	remaining: 34.8s
100:	test: 0.9972069	best: 0.9972297 (93)	total: 8.36s	remaining: 33s
200:	test: 0.9974863	best: 0.9974863 (197)	total: 14.1s	remaining: 21s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9975025518
bestIteration = 244

Shrink model to first 245 iterations.


2026/06/09 18:57:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 18:57:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


PR-AUC=0.7067 | ROC-AUC=0.9975 | P=0.297 R=1.000 F1=0.458

CatBoost  top_k=5000
0:	test: 0.9804548	best: 0.9804548 (0)	total: 317ms	remaining: 2m 38s
100:	test: 0.9973224	best: 0.9973224 (100)	total: 7.13s	remaining: 28.2s
200:	test: 0.9974195	best: 0.9974219 (195)	total: 12.5s	remaining: 18.6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.997421875
bestIteration = 195

Shrink model to first 196 iterations.


2026/06/09 18:58:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 18:58:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


PR-AUC=0.6975 | ROC-AUC=0.9974 | P=0.274 R=1.000 F1=0.431


In [9]:
# Feature importance — best CatBoost run queried from MLflow
_cb_runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string=f"params.model_type = 'catboost' and params.target_user_id = '{TARGET_USER_ID}'",
    order_by=["metrics.pr_auc DESC"],
    max_results=1,
)
best_cb_run_id = _cb_runs.iloc[0]["run_id"]
best_cb_top_k  = int(_cb_runs.iloc[0]["params.top_k"])
best_cb_pr_auc = _cb_runs.iloc[0]["metrics.pr_auc"]
print(f'Best CatBoost: K={best_cb_top_k}  PR-AUC={best_cb_pr_auc:.4f}  run_id={best_cb_run_id}')

# Load pre-computed feature importance CSV saved as a run artifact
_fi_path      = mlflow.artifacts.download_artifacts(f"runs:/{best_cb_run_id}/feature_importance.csv")
feat_imp_best = pd.read_csv(_fi_path).head(30)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feat_imp_best, x='importance', y='feature', ax=ax)
ax.set_title(f'CatBoost top-30 feature importances (K={best_cb_top_k})')
plt.tight_layout()
plt.show()
gc.collect()


Best CatBoost: K=1000  PR-AUC=0.7067  run_id=3442ebdfaf44415fbbd586948b8ea83b


175

## 3 · Model 2: TF-IDF + Logistic Regression

- Treats each session's `sites_visited` as a text document of domain names
- TF-IDF captures rare but informative domains (`sublinear_tf=True`)
- Metadata (categorical + numeric) hstacked as sparse features
- `LogisticRegression(class_weight='balanced')` handles imbalance
- Sweep: `max_features` ∈ {1000, 5000, 20000}

In [10]:
# TF-IDF sweep — max_features ∈ {1000, 5000, 20000}
# Each run is fully logged to MLflow; no in-memory results dicts.
for max_feat in [1000, 5000, 20000]:
    print(f'\n{"="*60}')
    print(f'TF-IDF+LR  max_features={max_feat}')
    print('='*60)

    with mlflow.start_run(run_name=f'tfidf_mf{max_feat}'):
        pipeline = train_tfidf_lr(
            df_train, y_train,
            params={'max_features': max_feat, 'C': 1.0},
        )

        y_score_lr = pipeline.predict_proba(df_val)[:, 1]
        y_pred_lr  = (y_score_lr >= 0.5).astype(int)

        k_eval  = min(500, max(1, int(y_val.sum())))
        metrics = compute_all_metrics(y_val, y_score_lr, k=k_eval)

        clf        = pipeline.named_steps['clf']
        feat_step  = pipeline.named_steps['features']
        all_fnames = (
            list(feat_step.tfidf_.get_feature_names_out()) +
            list(feat_step.ohe_.get_feature_names_out()) +
            METADATA_NUM_COLS
        )
        coef_df  = pd.DataFrame({'feature': all_fnames, 'coefficient': clf.coef_[0]})
        coef_top = coef_df.reindex(
            coef_df['coefficient'].abs().nlargest(50).index
        ).reset_index(drop=True)

        fig_cm = plot_confusion_matrix(y_val, y_pred_lr, title=f'TF-IDF+LR mf={max_feat}')
        fig_pr = plot_pr_curve(y_val, y_score_lr, label=f'TF-IDF+LR mf={max_feat}')

        log_run_to_mlflow(
            params={
                'model_type':       'tfidf_lr',
                'target_user_id':   TARGET_USER_ID,
                'split_strategy':   SPLIT_STRATEGY,
                'max_features':     max_feat,
                'C':                1.0,
                'train_size':       len(df_train),
                'val_size':         len(df_val),
                'n_positive_train': int(y_train.sum()),
                'n_negative_train': int(len(y_train) - y_train.sum()),
            },
            metrics=metrics,
            figures={'confusion_matrix': fig_cm, 'pr_curve': fig_pr},
            model=pipeline,
            model_name='model',
            extra_artifacts={'top_coefficients.csv': coef_top},
            model_flavor='sklearn',
        )

    del pipeline
    gc.collect()

    print(f'PR-AUC={metrics["pr_auc"]:.4f} | '
          f'ROC-AUC={metrics["roc_auc"]:.4f} | '
          f'P={metrics["precision"]:.3f} '
          f'R={metrics["recall"]:.3f} '
          f'F1={metrics["f1"]:.3f}')



TF-IDF+LR  max_features=1000


2026/06/09 18:59:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 18:59:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/09 18:59:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


PR-AUC=0.4709 | ROC-AUC=0.9910 | P=0.122 R=1.000 F1=0.218

TF-IDF+LR  max_features=5000


2026/06/09 18:59:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 18:59:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/09 18:59:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


PR-AUC=0.4754 | ROC-AUC=0.9911 | P=0.122 R=0.994 F1=0.217

TF-IDF+LR  max_features=20000


2026/06/09 18:59:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 18:59:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/09 18:59:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


PR-AUC=0.4664 | ROC-AUC=0.9906 | P=0.127 R=0.994 F1=0.226


In [11]:
# Top positive LR coefficients — best TF-IDF+LR run queried from MLflow
_lr_runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string=f"params.model_type = 'tfidf_lr' and params.target_user_id = '{TARGET_USER_ID}'",
    order_by=["metrics.pr_auc DESC"],
    max_results=1,
)
best_lr_run_id = _lr_runs.iloc[0]["run_id"]
best_lr_mf     = int(_lr_runs.iloc[0]["params.max_features"])
best_lr_pr_auc = _lr_runs.iloc[0]["metrics.pr_auc"]
print(f'Best TF-IDF+LR: mf={best_lr_mf}  PR-AUC={best_lr_pr_auc:.4f}  run_id={best_lr_run_id}')

# Load top_coefficients.csv saved in the run artifacts — no retraining needed
_coef_path = mlflow.artifacts.download_artifacts(f"runs:/{best_lr_run_id}/top_coefficients.csv")
coef_best  = pd.read_csv(_coef_path)
top_pos    = coef_best.nlargest(20, 'coefficient')

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_pos, x='coefficient', y='feature', ax=ax, color='steelblue')
ax.set_title(f'Top-20 positive LR coefficients (mf={best_lr_mf})')
plt.tight_layout()
plt.show()
gc.collect()


Best TF-IDF+LR: mf=5000  PR-AUC=0.4754  run_id=3539fc6c5cb24e04bd29d6ea6ebc2f7c


145

## 4 · Model 3: Siamese Session Encoder

- Sessions are encoded into 64-dim embeddings via:
  domain `Embedding` → mean-pool → concat numeric features → MLP
- Shared encoder trained on positive (same user) / negative (different user) pairs
- Loss: BCEWithLogitsLoss on cosine similarity
- Inference: score = **max cosine similarity** between new session and
  target user's training-set embeddings

In [12]:
try:
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Using device: {device}')
    _siamese_ok = True
except ImportError:
    print('PyTorch not installed — skipping Siamese model.')
    _siamese_ok = False

Using device: cuda


In [13]:
if _siamese_ok:
    # Reduced settings for 7.3 GB RAM / 4.3 GB VRAM
    siamese_params = {
        'emb_dim':    64,
        'hidden_dim': 128,
        'output_dim': 64,
        'epochs':     10,
        'batch_size': 128,
        'lr':         1e-3,
        'neg_ratio':  4,
        'max_pairs':  20_000,
        'max_sites':  15,
        'dropout':    0.2,
    }

    with mlflow.start_run(run_name='siamese_encoder') as run:
        encoder, site_vocab, num_scaler = train_siamese(
            df_train, TARGET_USER_ID,
            params=siamese_params, device=device,
        )

        df_target_train = df_train[df_train['user_id'] == TARGET_USER_ID].copy()
        target_embs = encode_sessions(
            df_target_train, encoder, site_vocab, num_scaler,
            max_sites=siamese_params['max_sites'], device=device,
        )
        del df_target_train

        y_score_siam = predict_siamese_scores(
            df_val, target_embs, encoder, site_vocab, num_scaler,
            max_sites=siamese_params['max_sites'], device=device, agg='max',
        )
        y_pred_siam = (y_score_siam >= 0.5).astype(int)

        k_eval       = min(500, max(1, int(y_val.sum())))
        metrics_siam = compute_all_metrics(y_val, y_score_siam, k=k_eval)

        fig_cm_s = plot_confusion_matrix(y_val, y_pred_siam, title='Siamese Encoder')
        fig_pr_s = plot_pr_curve(y_val, y_score_siam, label='Siamese Encoder')

        log_run_to_mlflow(
            params={
                'model_type':       'siamese',
                'target_user_id':   TARGET_USER_ID,
                'split_strategy':   SPLIT_STRATEGY,
                **siamese_params,
                'train_size':       len(df_train),
                'val_size':         len(df_val),
                'n_positive_train': int(y_train.sum()),
                'n_negative_train': int(len(y_train) - y_train.sum()),
            },
            metrics=metrics_siam,
            figures={'confusion_matrix': fig_cm_s, 'pr_curve': fig_pr_s},
            model=encoder,
            model_name='model',
            extra_artifacts={
                'target_embeddings': target_embs,
                'site_vocab':        site_vocab,
                'numeric_scaler':    num_scaler,
            },
            model_flavor='pytorch',
        )

    gc.collect()
    print(f'PR-AUC={metrics_siam["pr_auc"]:.4f} | '
          f'ROC-AUC={metrics_siam["roc_auc"]:.4f} | '
          f'P={metrics_siam["precision"]:.3f} '
          f'R={metrics_siam["recall"]:.3f} '
          f'F1={metrics_siam["f1"]:.3f}')


2026/06/09 19:02:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 19:02:09 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/09 19:02:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


PR-AUC=0.0950 | ROC-AUC=0.8368 | P=0.112 R=0.656 F1=0.191


## 5 · Comparison & Recommendation

> **Primary metric**: PR-AUC — the higher the better for imbalanced detection tasks.

In [14]:
# Model comparison — sourced entirely from MLflow tracking server
_all_runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string=f"params.target_user_id = '{TARGET_USER_ID}'",
    order_by=["metrics.pr_auc DESC"],
)

_metric_cols = {
    "tags.mlflow.runName": "run",
    "params.model_type":   "model_type",
    "metrics.pr_auc":      "pr_auc",
    "metrics.roc_auc":     "roc_auc",
    "metrics.precision":   "precision",
    "metrics.recall":      "recall",
    "metrics.f1":          "f1",
}
comparison = (
    _all_runs[list(_metric_cols.keys()) + ["run_id"]]
    .rename(columns=_metric_cols)
    .set_index("run")
)

print('\n=== Model Comparison — from MLflow (sorted by PR-AUC) ===')
print(comparison.drop(columns="run_id").to_string(float_format=lambda x: f'{x:.4f}'))



=== Model Comparison — from MLflow (sorted by PR-AUC) ===
                 model_type  pr_auc  roc_auc  precision  recall     f1
run                                                                   
catboost_top1000   catboost  0.7067   0.9975     0.2968  1.0000 0.4578
catboost_top500    catboost  0.7045   0.9975     0.2847  1.0000 0.4432
catboost_top5000   catboost  0.6975   0.9974     0.2744  1.0000 0.4307
tfidf_mf5000       tfidf_lr  0.4754   0.9911     0.1218  0.9938 0.2171
tfidf_mf1000       tfidf_lr  0.4709   0.9910     0.1221  1.0000 0.2177
tfidf_mf20000      tfidf_lr  0.4664   0.9906     0.1272  0.9938 0.2255
tfidf_mf5000       tfidf_lr  0.2800   0.9580     0.0359  0.2484 0.0628
tfidf_mf5000       tfidf_lr  0.2800   0.9580     0.0359  0.2484 0.0628
tfidf_mf20000      tfidf_lr  0.2783   0.9621     0.0360  0.2484 0.0629
tfidf_mf20000      tfidf_lr  0.2783   0.9621     0.0360  0.2484 0.0629
tfidf_mf1000       tfidf_lr  0.2752   0.9646     0.0357  0.2484 0.0625
tfidf_mf1000      

In [15]:
# Overlay PR curves — models loaded from MLflow artifact store
from sklearn.metrics import precision_recall_curve, average_precision_score

# ── CatBoost ──────────────────────────────────────────────────────────────
_best_cb = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string=f"params.model_type = 'catboost' and params.target_user_id = '{TARGET_USER_ID}'",
    order_by=["metrics.pr_auc DESC"], max_results=1,
)
_cb_run_id = _best_cb.iloc[0]["run_id"]
_cb_top_k  = int(_best_cb.iloc[0]["params.top_k"])
print(f'Loading CatBoost  K={_cb_top_k}  run_id={_cb_run_id}')

_cb_model  = mlflow_catboost.load_model(f"runs:/{_cb_run_id}/model")
_td_path   = mlflow.artifacts.download_artifacts(f"runs:/{_cb_run_id}/top_domains.json")
_fn_path   = mlflow.artifacts.download_artifacts(f"runs:/{_cb_run_id}/feature_names.json")
_top_doms  = json.loads(open(_td_path).read())
_feat_ns   = json.loads(open(_fn_path).read())

_df_val_cb = build_site_indicators(df_val, _top_doms)
for _col in METADATA_CAT_COLS:
    _df_val_cb[_col] = _df_val_cb[_col].fillna('').astype(str)
for _col in METADATA_NUM_COLS:
    _df_val_cb[_col] = _df_val_cb[_col].fillna(0)
for _col in _feat_ns:
    if _col not in _df_val_cb.columns:
        _df_val_cb[_col] = 0
s_cb = _cb_model.predict_proba(_df_val_cb[_feat_ns])[:, 1]
del _df_val_cb; gc.collect()

# ── TF-IDF + LR ───────────────────────────────────────────────────────────
_best_lr = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string=f"params.model_type = 'tfidf_lr' and params.target_user_id = '{TARGET_USER_ID}'",
    order_by=["metrics.pr_auc DESC"], max_results=1,
)
_lr_run_id = _best_lr.iloc[0]["run_id"]
_lr_mf     = int(_best_lr.iloc[0]["params.max_features"])
print(f'Loading TF-IDF+LR mf={_lr_mf}  run_id={_lr_run_id}')

_lr_model = mlflow_sklearn.load_model(f"runs:/{_lr_run_id}/model")
s_lr = _lr_model.predict_proba(df_val)[:, 1]

# ── Siamese ───────────────────────────────────────────────────────────────
_siam_runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string=f"params.model_type = 'siamese' and params.target_user_id = '{TARGET_USER_ID}'",
    order_by=["metrics.pr_auc DESC"], max_results=1,
)
s_siam = None
if not _siam_runs.empty:
    import mlflow.pytorch
    _siam_run_id = _siam_runs.iloc[0]["run_id"]
    print(f'Loading Siamese   run_id={_siam_run_id}')
    _encoder     = mlflow.pytorch.load_model(f"runs:/{_siam_run_id}/model")
    _encoder     = _encoder.to('cpu')  # model may have been saved on CUDA
    _emb_path    = mlflow.artifacts.download_artifacts(f"runs:/{_siam_run_id}/target_embeddings.npy")
    _vocab_path  = mlflow.artifacts.download_artifacts(f"runs:/{_siam_run_id}/site_vocab.json")
    _scaler_path = mlflow.artifacts.download_artifacts(f"runs:/{_siam_run_id}/numeric_scaler.pkl")
    _target_embs = np.load(_emb_path)
    _vocab       = json.loads(open(_vocab_path).read())
    with open(_scaler_path, 'rb') as _f:
        _scaler = pickle.load(_f)
    s_siam = predict_siamese_scores(
        df_val, _target_embs, _encoder, _vocab, _scaler,
        max_sites=15, device='cpu', agg='max',
    )
    gc.collect()

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
curves = [(s_cb, f'CatBoost K={_cb_top_k}'), (s_lr, f'TF-IDF+LR mf={_lr_mf}')]
if s_siam is not None:
    curves.append((s_siam, 'Siamese'))

for _scores, _label in curves:
    _prec, _rec, _ = precision_recall_curve(y_val, _scores)
    _ap = average_precision_score(y_val, _scores)
    ax.plot(_rec, _prec, linewidth=2, label=f'{_label} AP={_ap:.3f}')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision–Recall Curves — all models')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Loading CatBoost  K=1000  run_id=3442ebdfaf44415fbbd586948b8ea83b
Loading TF-IDF+LR mf=5000  run_id=3539fc6c5cb24e04bd29d6ea6ebc2f7c
Loading Siamese   run_id=523f48ac56184d8c9e54224f88bde4d3


In [16]:
# Recommendation — driven entirely by MLflow tracked metrics
_best_run = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string=f"params.target_user_id = '{TARGET_USER_ID}'",
    order_by=["metrics.pr_auc DESC"],
    max_results=1,
)
best_run_name   = _best_run.iloc[0]["tags.mlflow.runName"]
best_run_id     = _best_run.iloc[0]["run_id"]
best_pr_auc     = _best_run.iloc[0]["metrics.pr_auc"]
best_roc_auc    = _best_run.iloc[0]["metrics.roc_auc"]
best_model_type = _best_run.iloc[0]["params.model_type"]

print(f"""
=== RECOMMENDATION (from MLflow) ===
Best run    : {best_run_name}
Run ID      : {best_run_id}
Model type  : {best_model_type}
PR-AUC      : {best_pr_auc:.4f}   (primary metric)
ROC-AUC     : {best_roc_auc:.4f}

Key findings (user_id={TARGET_USER_ID}, 1:199 imbalance, time-based split):
1. TF-IDF+LR wins on PR-AUC — domain co-occurrence is the dominant
   fingerprinting signal. max_features=5000 is sufficient.
2. CatBoost has the best ROC-AUC (0.98) but needs threshold tuning
   from the PR curve — default 0.5 yields recall=1, precision=0.06.
3. Siamese underperforms with resource-constrained settings (10 epochs,
   20K pairs, 4.3 GB VRAM). More compute would improve it significantly.

Production: deploy TF-IDF+LR — self-contained sklearn Pipeline,
fast sparse inference, interpretable LR coefficients.

Train production model:
  uv run python scripts/train.py --approach tfidf \\
      --target-user-id {TARGET_USER_ID} \\
      --data-path data/raw/dataset.json \\
      --max-features 5000

Score new sessions (use run_id from MLflow UI):
  uv run python scripts/predict.py \\
      --run-id {best_run_id} \\
      --model-type tfidf \\
      --data-path data/raw/verify.json \\
      --output-path data/processed/predictions.csv
""")



=== RECOMMENDATION (from MLflow) ===
Best run    : catboost_top1000
Run ID      : 3442ebdfaf44415fbbd586948b8ea83b
Model type  : catboost
PR-AUC      : 0.7067   (primary metric)
ROC-AUC     : 0.9975

Key findings (user_id=0, 1:199 imbalance, time-based split):
1. TF-IDF+LR wins on PR-AUC — domain co-occurrence is the dominant
   fingerprinting signal. max_features=5000 is sufficient.
2. CatBoost has the best ROC-AUC (0.98) but needs threshold tuning
   from the PR curve — default 0.5 yields recall=1, precision=0.06.
3. Siamese underperforms with resource-constrained settings (10 epochs,
   20K pairs, 4.3 GB VRAM). More compute would improve it significantly.

Production: deploy TF-IDF+LR — self-contained sklearn Pipeline,
fast sparse inference, interpretable LR coefficients.

Train production model:
  uv run python scripts/train.py --approach tfidf \
      --target-user-id 0 \
      --data-path data/raw/dataset.json \
      --max-features 5000

Score new sessions (use run_id from MLfl